Simple Circuit File Reader - Read QASM files into QuantumCircuit objects

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(f"./")
sys.path.append(f"./../")

In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(f"./")
sys.path.append(f"./../")

import os
import glob
import re
from typing import Dict, List, Optional, Tuple
from qiskit import QuantumCircuit

from src.misc import multi_crx


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def find_debug_files():
    """Find debug files, newest first."""
    files = glob.glob("./../mcrx_optimization_errors/mcrx_optimization_error_*.txt")
    return sorted(files, key=os.path.getmtime, reverse=True)

def parse_circuit_from_debug(filepath: str, circuit_type: str = "original") -> QuantumCircuit:
    """Parse circuit from debug file by directly extracting gate info."""
    with open(filepath, 'r') as f:
        content = f.read()
    
    lines = content.split('\n')
    
    # Find circuit section
    start_marker = f"{circuit_type.upper()} CIRCUIT:"
    start_idx = None
    for i, line in enumerate(lines):
        if start_marker in line:
            start_idx = i
            break
    
    if start_idx is None:
        return QuantumCircuit(1)
    
    # Find number of qubits
    num_qubits = 1
    for line in lines[start_idx:start_idx+10]:
        if "Total qubits:" in line:
            num_qubits = int(line.split(":")[1].strip())
            break
    
    circuit = QuantumCircuit(num_qubits)
    
    # Find gates section
    gates_start = None
    for i in range(start_idx, len(lines)):
        if lines[i].strip() == "Gates:":
            gates_start = i + 1
            break
    
    if gates_start is None:
        return circuit
    
    # Parse each gate
    i = gates_start
    while i < len(lines):
        line = lines[i].strip()
        
        # Stop at next section
        if not line or line.startswith("DETAILED") or line.startswith("OPTIMIZED"):
            break
        
        # Process gate lines
        if re.match(r'^\d+\.', line):
            # Parse this gate and its info
            gate_lines = [line]
            j = i + 1
            # Collect all lines for this gate
            while j < len(lines) and (lines[j].startswith('   ') or lines[j].strip() == ''):
                if lines[j].strip():
                    gate_lines.append(lines[j])
                j += 1
            
            # Add gate to circuit
            add_gate_from_lines(circuit, gate_lines)
            i = j
        else:
            i += 1
    
    return circuit

def add_gate_from_lines(circuit, gate_lines):
    """Add gate to circuit from its lines."""
    if not gate_lines:
        return
    
    # Parse main gate line
    gate_line = gate_lines[0].strip()
    
    # Extract gate name and angle
    if '(' in gate_line and ')' in gate_line:
        # Has angle: "1. c8rx_o253(0.7854)"
        match = re.search(r'\d+\.\s+(\w+(?:_\w+)*)\(([0-9.-]+)\)', gate_line)
        if match:
            gate_name = match.group(1)
            angle = float(match.group(2))
        else:
            return
    else:
        # No angle: "2. cx"
        match = re.search(r'\d+\.\s+(\w+)', gate_line)
        if match:
            gate_name = match.group(1)
            angle = 0.0
        else:
            return
    
    # Extract info from other lines
    qubits = []
    controls = []
    target = None
    control_state = None
    
    for line in gate_lines[1:]:
        line = line.strip()
        
        # Get qubits
        if "Qubits:" in line:
            match = re.search(r'Qubits:\s*\[(.*?)\]', line)
            if match:
                qubits = [int(x.strip()) for x in match.group(1).split(',')]
        
        # Get controls and target
        elif "Controls:" in line and "Target:" in line:
            match = re.search(r'Controls:\s*\[(.*?)\]\s*->\s*Target:\s*(\d+)', line)
            if match:
                controls = [int(x.strip()) for x in match.group(1).split(',') if x.strip()]
                target = int(match.group(2))
        
        # Get control and target (CX style)
        elif "Control:" in line and "Target:" in line:
            match = re.search(r'Control:\s*(\d+)\s*->\s*Target:\s*(\d+)', line)
            if match:
                controls = [int(match.group(1))]
                target = int(match.group(2))
                qubits = [controls[0], target]
        
        # Get target only
        elif "Target:" in line and "Control" not in line:
            match = re.search(r'Target:\s*(\d+)', line)
            if match:
                target = int(match.group(1))
                if not qubits:
                    qubits = [target]
        
        # Get control state
        elif "Control state:" in line:
            match = re.search(r'Control state:\s*(\d+)', line)
            if match:
                control_state = int(match.group(1))
    
    # Add gate to circuit
    clean_name = gate_name.lower()
    
    try:
        if clean_name == 'x':
            circuit.x(qubits[0])
        
        elif clean_name in ['cx', 'cnot']:
            circuit.cx(qubits[0], qubits[1])
        
        elif clean_name == 'rx':
            circuit.rx(angle, qubits[0])
        
        elif 'rx' in clean_name and controls:
            # MCRX gate - convert control state to pattern
            if control_state is not None:
                pattern = bin(control_state)[2:]  # Remove '0b'
                pattern = pattern.zfill(len(controls))  # Pad zeros
                pattern = pattern[::-1]  # Reverse for left-to-right
            else:
                pattern = '1' * len(controls)
            
            gate = multi_crx(angle, pattern)
            circuit.append(gate, qubits)
        
        print(f"Added {clean_name} gate on qubits {qubits}")
        
    except Exception as e:
        print(f"Error adding {clean_name}: {e}")

def get_circuits_from_file(filepath: str):
    """Get circuits from a specific debug file."""
    print(f"Processing: {os.path.basename(filepath)}")
    
    original = parse_circuit_from_debug(filepath, "original")
    optimized = parse_circuit_from_debug(filepath, "optimized")
    
    return original, optimized

def get_latest_circuits():
    """Get circuits from latest debug file."""
    files = find_debug_files()
    if not files:
        print("No debug files found")
        return None, None
    
    filepath = files[0]
    return get_circuits_from_file(filepath)

def analyze_file(filepath: str):
    """Analyze a specific debug file."""
    print(f"\n{'='*60}")
    print(f"FILE: {os.path.basename(filepath)}")
    print(f"{'='*60}")
    
    original, optimized = get_circuits_from_file(filepath)
    
    if original and len(original.data) > 0:
        print(f"\nOriginal: {len(original.data)} gates, {original.num_qubits} qubits")
        print(original.draw())
    else:
        print("\nNo original circuit found")
    
    print(f"\n{'-'*30}")
    if optimized and len(optimized.data) > 0:
        print(f"Optimized: {len(optimized.data)} gates, {optimized.num_qubits} qubits")
        print(optimized.draw())
    else:
        print("No optimized circuit found")
    
    if original and optimized and len(original.data) > 0 and len(optimized.data) > 0:
        reduction = len(original.data) - len(optimized.data)
        reduction_pct = (reduction / len(original.data)) * 100 if len(original.data) > 0 else 0
        print(f"\nGate reduction: {reduction} gates ({reduction_pct:.1f}%)")
    
    return original, optimized

def analyze_latest():
    """Quick analysis of latest debug file."""
    files = find_debug_files()
    if not files:
        print("No debug files found")
        return None, None
    
    return analyze_file(files[0])

def analyze_all():
    """Analyze all debug files."""
    files = find_debug_files()
    if not files:
        print("No debug files found")
        return []
    
    print(f"Found {len(files)} debug files to analyze:")
    for i, filepath in enumerate(files):
        print(f"{i+1}. {os.path.basename(filepath)}")
    
    results = []
    for filepath in files:
        try:
            original, optimized = analyze_file(filepath)
            results.append((filepath, original, optimized))
        except Exception as e:
            print(f"Error processing {os.path.basename(filepath)}: {e}")
    
    # Summary
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL FILES")
    print(f"{'='*60}")
    
    for i, (filepath, original, optimized) in enumerate(results):
        filename = os.path.basename(filepath)
        orig_gates = len(original.data) if original else 0
        opt_gates = len(optimized.data) if optimized else 0
        reduction = orig_gates - opt_gates
        reduction_pct = (reduction / orig_gates) * 100 if orig_gates > 0 else 0
        
        print(f"{i+1:2d}. {filename}")
        print(f"    Original: {orig_gates:3d} gates | Optimized: {opt_gates:3d} gates | Reduction: {reduction:3d} ({reduction_pct:5.1f}%)")
    
    return results


In [4]:
# Auto-run - now analyzes ALL files instead of just the latest
files = find_debug_files()
if files:
    print(f"Found {len(files)} debug files")
    analyze_all()
else:
    print("No debug files found")

Found 980 debug files
Found 980 debug files to analyze:
1. mcrx_optimization_error_20250619_203442.txt
2. mcrx_optimization_error_20250619_203441.txt
3. mcrx_optimization_error_20250619_203440.txt
4. mcrx_optimization_error_20250619_203439.txt
5. mcrx_optimization_error_20250619_203438.txt
6. mcrx_optimization_error_20250619_203437.txt
7. mcrx_optimization_error_20250619_203436.txt
8. mcrx_optimization_error_20250619_203435.txt
9. mcrx_optimization_error_20250619_203434.txt
10. mcrx_optimization_error_20250619_203433.txt
11. mcrx_optimization_error_20250619_203432.txt
12. mcrx_optimization_error_20250619_203431.txt
13. mcrx_optimization_error_20250619_203430.txt
14. mcrx_optimization_error_20250619_203429.txt
15. mcrx_optimization_error_20250619_203428.txt
16. mcrx_optimization_error_20250619_203427.txt
17. mcrx_optimization_error_20250619_203426.txt
18. mcrx_optimization_error_20250619_203425.txt
19. mcrx_optimization_error_20250619_203424.txt
20. mcrx_optimization_error_20250619_2034